# Preprocessing: resize Plant Pathology 2021 images → 512px

Run this notebook **once** (CPU, Internet can be Off). It downsizes every train/test
image so that the longest side is `MAX_SIDE` px and re-saves them as JPEG.

After the run:
1. **Save Version** (Save & Run All) — the contents of `/kaggle/working` become the output.
2. On the notebook page: **Output → New Dataset** (or create a dataset from the version).
3. In the training notebook: **Add Data** this dataset and set
   `CFG.DATA_DIR = Path('/kaggle/input/<your-slug>')`.

Large JPEGs (~4000px) are the main training bottleneck: reading them from disk and
decoding eats all the time while the GPU sits idle. 512px files are many times smaller →
epochs run many times faster.

In [ ]:
import shutil
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

from PIL import Image
from tqdm.auto import tqdm

SRC_DIR = Path("/kaggle/input/competitions/plant-pathology-2021-fgvc8")
DST_DIR = Path("/kaggle/working/plant-pathology-512")
MAX_SIDE = 512        # longest side of the result
JPEG_QUALITY = 90
NUM_WORKERS = 4       # Kaggle usually has 4 vCPUs

SUBDIRS = ["train_images", "test_images"]
EXTS = {".jpg", ".jpeg", ".png"}

In [ ]:
def resize_one(args):
    src_path, dst_path = args
    try:
        image = Image.open(src_path)
        # draft speeds up decoding of the large source JPEG itself
        image.draft("RGB", (MAX_SIDE, MAX_SIDE))
        image = image.convert("RGB")
        w, h = image.size
        scale = MAX_SIDE / max(w, h)
        if scale < 1.0:
            image = image.resize((round(w * scale), round(h * scale)), Image.BILINEAR)
        image.save(dst_path, format="JPEG", quality=JPEG_QUALITY)
        return True
    except Exception as exc:
        print(f"FAILED {src_path}: {exc}")
        return False


def build_jobs():
    jobs = []
    for sub in SUBDIRS:
        src_sub = SRC_DIR / sub
        dst_sub = DST_DIR / sub
        dst_sub.mkdir(parents=True, exist_ok=True)
        for p in src_sub.iterdir():
            if p.suffix.lower() in EXTS:
                jobs.append((p, dst_sub / (p.stem + ".jpg")))
    return jobs

In [ ]:
DST_DIR.mkdir(parents=True, exist_ok=True)

# Copy CSV files as-is
for name in ["train.csv", "sample_submission.csv"]:
    src = SRC_DIR / name
    if src.exists():
        shutil.copy(src, DST_DIR / name)
        print(f"copied {name}")

jobs = build_jobs()
print(f"Total images: {len(jobs)}")

ok = 0
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as ex:
    for res in tqdm(ex.map(resize_one, jobs, chunksize=16), total=len(jobs)):
        ok += int(res)

print(f"Done: {ok}/{len(jobs)} saved to {DST_DIR}")

In [ ]:
# Quick sanity check of the result
for sub in SUBDIRS:
    files = list((DST_DIR / sub).glob("*.jpg"))
    print(f"{sub}: {len(files)} files")
    if files:
        sample = Image.open(files[0])
        size_kb = files[0].stat().st_size / 1024
        print(f"  sample: {files[0].name} | {sample.size} | {size_kb:.0f} KB")